# 02 — ACS Structural Feature Layer (Individuals)

## Objective

Build the structural baseline segmentation layer for the U.S. population using ACS PUMS (PUS — individuals).

This layer represents the demographic and socioeconomic architecture of U.S. society, independent of psychographics and media behavior (to be integrated later).

---

## Scope of This Notebook

This notebook:

1. Loads PUS Parquet shards (interim layer).
2. Defines a controlled structural feature schema (v1).
3. Selects and cleans relevant variables.
4. Applies a weighting strategy.
5. Creates a representative modeling sample (~2M individuals).
6. Produces the first structural modeling table.

Clustering is not performed in this notebook.  
This stage prepares the modeling matrix.

---

## Segmentation Unit

**Unit of analysis:** Individual (PUS)

Rationale:
- Behavioral targeting operates at the person level.
- GSS and NPORS are person-level datasets.
- Enables probabilistic fusion in later stages.

---

## Modeling Strategy

- Person weight (`PWGTP`) is preserved for representativeness.
- A weighted modeling sample is created for computational efficiency.
- Full population weights are retained for later evaluation.

The full ~15M rows are not used for clustering.

---

## Structural Feature Domains (v1 Draft)

### 1. Demographics
- Age
- Sex
- Race
- Hispanic origin
- Marital status

### 2. Socioeconomic Status
- Education
- Personal income
- Employment status
- Occupation class

### 3. Geography
- Region (derived)
- Urban/rural proxy (if available)

### 4. Household Context (individual-level fields only)
- Household income
- Household size (if available in PUS)

---

### Population Scope Decision

The ACS PUMS structural layer includes the full U.S. population (all ages).

Although downstream psychographic datasets (e.g., NPORS, GSS, Pew surveys) are restricted to U.S. adults (18+), minors (<18) are retained in the structural dataset for the following reasons:

1. Minors may function as indirect Target Audiences (TA) capable of influencing adult decision-makers (e.g., parents).
2. Household-level targeting often depends on child presence and age structure.
3. Removing minors would distort structural national priors.

To preserve methodological clarity, we introduce an `actor_class` variable:

- Adult (AGEP >= 18)
- Minor (AGEP < 18)

Psychographic fusion layers will be applied only to the Adult subpopulation.

---
### Age Binning Strategy

The Structural Feature Layer uses marketing-style age bins to preserve
segmentation granularity and tactical flexibility.

Bins:
0–5
6–12
13–17
18–24
25–34
35–44
45–54
55–64
65+

These bins align with common media planning and performance marketing
standards.

A derived Pew-aligned binning (18–29, 30–49, 50–64, 65+) will be added
later when integrating psychographic/media consumption datasets
(e.g., NPORS, GSS).

---
### Race / Ethnicity Harmonization

Race (`RAC1P`) and Hispanic origin (`HISP`) are combined into a single
harmonized variable `race_eth`.

Classification logic:

- Hispanic (any race)
- White_NH (Non-Hispanic White)
- Black_NH (Non-Hispanic Black)
- Asian_NH (Non-Hispanic Asian)
- Other_NH (Non-Hispanic all other races)

This structure aligns with Census reporting and major survey datasets
(e.g., Pew, GSS) for cross-layer compatibility.

---

### Education Tier Harmonization

The ACS variable `SCHL` contains detailed educational attainment codes.

For structural modeling and segmentation purposes, we collapse these
into four macro tiers:

- HS_or_less
- Some_college
- Bachelor
- Graduate

This preserves socioeconomic signal while avoiding excessive sparsity.

---
### Employment Tier Harmonization

The ACS variable `ESR` (Employment Status Recode) is collapsed into
macro employment tiers that reflect full labor-force structure:

- Employed
- Unemployed
- Retired
- Student
- Other_Not_in_Labor_Force

This preserves structural labor participation patterns while remaining
interpretable for segmentation and modeling.

---
### Income Tier Definition (Adult-Only)

Income tiers (`income_tier_fixed` and `income_tier_pct`) are defined only for Adults (AGEP ≥ 18).  
Minors are assigned NA because personal income (PINCP) reflects individual economic agency, which is not conceptually applicable to dependent children.

This avoids conflating non-earners (e.g., minors) with low-income adults and preserves structural interpretability for segmentation and modeling.

---

### Commute Mode Harmonization (JWTRNS)

The ACS variable `JWTRNS` (Journey to Work – Transportation Mode) is recoded into a simplified `commute_tier` variable:

- Car  
- Public_Transit  
- Walk  
- Work_From_Home  
- Other  

This variable serves as a lifestyle and urbanicity proxy and is behaviorally relevant for media exposure, daily routines, and segmentation modeling.

Commute mode is defined for the full population; individuals for whom commuting is not applicable (e.g., non-workers, minors) retain NA.

---


## Design Principles

- Features must be interpretable.
- Categories should reflect meaningful structural divisions within U.S. society.
- Recoding decisions must be documented.
- Protected attributes are not used for discriminatory modeling purposes.

---

## Deliverable of This Notebook

A clean DataFrame:

`acs_structural_model_v1`

Containing:
- Selected features
- Cleaned categories
- Person weights
- Ready for clustering in Notebook 03

In [20]:
# imports
# Core
import pandas as pd
import numpy as np
import os
import glob

# Visualization (minimal)
import matplotlib.pyplot as plt

# Sklearn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [21]:
# data paths
BASE_PATH = "/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y"

In [22]:
# Loading PERSON data
# Find all person parquet files (psam_pus*.parquet)
person_files = sorted(
    glob.glob(os.path.join(BASE_PATH, "psam_pus*.parquet"))
)

person_files[:5], len(person_files)

(['/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusa.parquet',
  '/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusb.parquet',
  '/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusc.parquet',
  '/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusd.parquet'],
 4)

In [23]:
# selecting relevent features only to speed up loading and processing
PERSON_COLS = [
    "SERIALNO",   # household link
    "SPORDER",    # person order in household
    
    "PWGTP",      # person weight
    
    # Demographic
    "SEX",
    "AGEP",
    "RAC1P",
    "HISP",
    "MAR",
    
    # Socioeconomic
    "SCHL",
    "ESR",
    "OCCP",
    "PINCP",
    "POVPIP",
    "JWTRNS",
    
    # Geography anchor
    #"ST",  #was not available in pusa. so removed for now
    "PUMA"
]

In [24]:
# build and save consloidated person dataframe
import gc
PERSON_OUT = os.path.join(BASE_PATH, "_consolidated_person.parquet")

df_person_list = []
for f in person_files:
    # Column-pruned read
    tmp = pd.read_parquet(f, columns=PERSON_COLS)
    df_person_list.append(tmp)

df_person = pd.concat(df_person_list, ignore_index=True)

# Optional sanity checks
assert "PWGTP" in df_person.columns
print(df_person.shape)
print(df_person["PWGTP"].describe())

# Save consolidated
df_person.to_parquet(PERSON_OUT, index=False)
print("Saved:", PERSON_OUT)

# Free memory aggressively
del df_person, df_person_list, tmp
gc.collect()

(15912393, 15)
count    1.591239e+07
mean     2.088860e+01
std      2.181002e+01
min      1.000000e+00
25%      9.000000e+00
50%      1.500000e+01
75%      2.400000e+01
max      1.071000e+03
Name: PWGTP, dtype: float64
Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/_consolidated_person.parquet


0

In [25]:
# Reading HOUSEHOLD data
housing_files = sorted(glob.glob(os.path.join(BASE_PATH, "psam_hus*.parquet")))
len(housing_files), housing_files[:5]

(4,
 ['/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husa.parquet',
  '/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husb.parquet',
  '/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husc.parquet',
  '/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husd.parquet'])

In [26]:
# selecting the relevant columns for housing data
HOUSING_COLS = [
    "SERIALNO",
    "WGTP",
    "TEN",      # tenure (own/rent)
    "HINCP",    # household income (if present)
    "NP",       # number of persons in household (if present)
    "BDSP",     # bedrooms (if present)
    "VEH",      # vehicles (if present)
    "FINCP",    # family income (if present)
    "GRNTP",    # gross rent (if present)
    "SMOCP",    # selected monthly owner costs (if present)
    "ST", "PUMA"
]

In [27]:
# making sure the selected columns are present in the housing files
sample_cols = set(pd.read_parquet(housing_files[0]).columns)
HOUSING_COLS = [c for c in HOUSING_COLS if c in sample_cols]
HOUSING_COLS

['SERIALNO',
 'WGTP',
 'TEN',
 'HINCP',
 'NP',
 'BDSP',
 'VEH',
 'FINCP',
 'GRNTP',
 'SMOCP',
 'PUMA']

In [28]:
# build and save consloidated housing dataframe
HOUSING_OUT = os.path.join(BASE_PATH, "_consolidated_housing.parquet")

df_housing_list = []
for f in housing_files:
    tmp = pd.read_parquet(f, columns=HOUSING_COLS)
    df_housing_list.append(tmp)

df_housing = pd.concat(df_housing_list, ignore_index=True)

assert "WGTP" in df_housing.columns
print(df_housing.shape)
print(df_housing["WGTP"].describe())

df_housing.to_parquet(HOUSING_OUT, index=False)
print("Saved:", HOUSING_OUT)

del df_housing, df_housing_list, tmp
gc.collect()

(7650159, 11)
count    7.650159e+06
mean     1.860522e+01
std      2.108395e+01
min      0.000000e+00
25%      7.000000e+00
50%      1.300000e+01
75%      2.300000e+01
max      9.120000e+02
Name: WGTP, dtype: float64
Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/_consolidated_housing.parquet


0

### Structural Features for PERSON dataset

In [29]:
# load person dataset for feature engineering

BASE_PATH = "/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y"
PERSON_PATH = os.path.join(BASE_PATH, "_consolidated_person.parquet")


df = pd.read_parquet(PERSON_PATH)
df.shape

(15912393, 15)

In [30]:
df.head(4)

,SERIALNO,SPORDER,PWGTP,SEX,AGEP,RAC1P,HISP,MAR,SCHL,ESR,OCCP,PINCP,POVPIP,JWTRNS,PUMA
0,2019GQ0000088,1,2,1,39,2,1,5,13.0,6.0,NaN,9000.0,68.0,NaN,2600
1,2019GQ0000096,1,14,2,21,1,1,5,13.0,6.0,NaN,150.0,NaN,NaN,700
2,2019GQ0000153,1,4,1,19,2,1,5,19.0,1.0,5240.0,1400.0,NaN,1.0,800
3,2019GQ0000198,1,17,1,77,1,1,2,12.0,6.0,NaN,22700.0,NaN,NaN,800


In [31]:
assert "PWGTP" in df.columns
df["PWGTP"].describe()

count    1.591239e+07
mean     2.088860e+01
std      2.181002e+01
min      1.000000e+00
25%      9.000000e+00
50%      1.500000e+01
75%      2.400000e+01
max      1.071000e+03
Name: PWGTP, dtype: float64

In [32]:
# creating actor class
df["actor_class"] = np.where(df["AGEP"] >= 18, "Adult", "Minor")

df["actor_class"].value_counts()

actor_class
Adult    12853377
Minor     3059016
Name: count, dtype: int64

In [33]:
# Compute weighted share
def weighted_share(data, col, weight="PWGTP"):
    totals = (
        data.groupby(col)[weight]
            .sum()
            .sort_values(ascending=False)
    )
    return (totals / totals.sum())

actor_distribution = weighted_share(df, "actor_class")
(actor_distribution * 100).round(2)

actor_class
Adult    77.85
Minor    22.15
Name: PWGTP, dtype: float64

In [35]:
# creating age bins
age_bins = [0, 6, 13, 18, 25, 35, 45, 55, 65, 120]
age_labels = [
    "0-5",
    "6-12",
    "13-17",
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

df["age_bin"] = pd.cut(
    df["AGEP"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

df["age_bin"].value_counts().sort_index()

age_bin
0-5       904550
6-12     1204736
13-17     949730
18-24    1358103
25-34    1889511
35-44    1899245
45-54    1912014
55-64    2293256
65+      3501248
Name: count, dtype: int64

In [36]:
# weighted age distribution
def weighted_distribution(data, col, weight="PWGTP"):
    s = (
        data.groupby(col)[weight]
            .sum()
            .sort_index()
    )
    return s / s.sum()

age_distribution = weighted_distribution(df, "age_bin")
(age_distribution * 100).round(2)

age_bin
0-5       6.86
6-12      8.71
13-17     6.58
18-24     9.13
25-34    13.67
35-44    13.10
45-54    12.29
55-64    12.82
65+      16.84
Name: PWGTP, dtype: float64

In [37]:
# sex recoding to match 1=Male, 2=Female
df["sex_label"] = df["SEX"].map({
    1: "Male",
    2: "Female"
})

df["sex_label"].value_counts()

sex_label
Female    8092621
Male      7819772
Name: count, dtype: int64

In [38]:
# weighted sex distribution
sex_distribution = (
    df.groupby("sex_label")["PWGTP"]
      .sum()
)

sex_distribution = sex_distribution / sex_distribution.sum()
(sex_distribution * 100).round(2)

sex_label
Female    50.5
Male      49.5
Name: PWGTP, dtype: float64

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15912393 entries, 0 to 15912392
Data columns (total 18 columns):
 #   Column       Dtype   
---  ------       -----   
 0   SERIALNO     str     
 1   SPORDER      int64   
 2   PWGTP        int64   
 3   SEX          int64   
 4   AGEP         int64   
 5   RAC1P        int64   
 6   HISP         int64   
 7   MAR          int64   
 8   SCHL         float64 
 9   ESR          float64 
 10  OCCP         float64 
 11  PINCP        float64 
 12  POVPIP       float64 
 13  JWTRNS       float64 
 14  PUMA         int64   
 15  actor_class  str     
 16  age_bin      category
 17  sex_label    str     
dtypes: category(1), float64(6), int64(8), str(3)
memory usage: 2.4 GB


In [40]:
# Education level recoding

#checking data missingness for education level
df["SCHL"].isna().mean()

np.float64(0.027185100317720912)

In [46]:
def recode_education(schl):
    if pd.isna(schl):
        return pd.NA
    
    schl = int(schl)
    
    if schl <= 17:
        return "HS_or_less"
    elif 18 <= schl <= 20:
        return "Some_college"
    elif schl == 21:
        return "Bachelor"
    else:
        return "Graduate"

df["edu_tier"] = df["SCHL"].apply(recode_education)

In [47]:
df["edu_tier"].value_counts()

edu_tier
HS_or_less      7439797
Some_college    3799910
Bachelor        2574554
Graduate        1665552
Name: count, dtype: int64

In [48]:
edu_distribution = (
    df.groupby("edu_tier")["PWGTP"]
      .sum()
)

edu_distribution = edu_distribution / edu_distribution.sum()
(edu_distribution * 100).round(2)

# don't forget, at this stage, even kids under 18 are included in the distribution, 
# which may skew the education distribution towards lower levels. 
# In future iterations, we might want to filter for adults only when analyzing education levels.

edu_tier
Bachelor        16.22
Graduate         9.83
HS_or_less      50.04
Some_college    23.91
Name: PWGTP, dtype: float64

In [49]:
weighted_share(df[df["actor_class"] == "Adult"], "edu_tier")

edu_tier
HS_or_less      0.379792
Some_college    0.296633
Bachelor        0.201503
Graduate        0.122073
Name: PWGTP, dtype: float64

In [50]:
# labor force participation recoding
df["ESR"].value_counts().sort_index()

ESR
1.0    7167499
2.0     150122
3.0     366707
4.0      63260
5.0        596
6.0    5489057
Name: count, dtype: int64

In [51]:
def recode_employment(row):
    esr = row["ESR"]
    age = row["AGEP"]
    
    if pd.isna(esr):
        return pd.NA
    
    esr = int(esr)
    
    # Employed (civilian + armed forces)
    if esr in (1, 2, 4, 5):
        return "Employed"
    
    # Unemployed
    if esr == 3:
        return "Unemployed"
    
    # Not in labor force (ESR == 6)
    if esr == 6:
        if age >= 65:
            return "Retired"
        elif age < 25:
            return "Student"
        else:
            return "Other_Not_in_Labor_Force"
    
    return pd.NA

df["emp_tier"] = df.apply(recode_employment, axis=1)

In [52]:
# weighted workforce distribution
emp_distribution = (
    df.groupby("emp_tier")["PWGTP"]
      .sum()
)

emp_distribution = emp_distribution / emp_distribution.sum()
(emp_distribution * 100).round(2)

emp_tier
Employed                    60.18
Other_Not_in_Labor_Force    13.63
Retired                     16.97
Student                      5.94
Unemployed                   3.28
Name: PWGTP, dtype: float64

In [53]:
# quick diagnostic of employment distribution in adults only
(weighted_distribution(
    df[df["actor_class"] == "Adult"],
    "emp_tier"
) * 100).round(2)

emp_tier
Employed                    61.43
Other_Not_in_Labor_Force    14.09
Retired                     17.54
Student                      3.68
Unemployed                   3.27
Name: PWGTP, dtype: float64

In [54]:
# personal income recoding
# check first the % of non-positive incomes

# 1) Flag non-positive income
df["nonpos_income"] = df["PINCP"].fillna(0) <= 0

# 2) Weighted share of non-positive income
w_nonpos = df.loc[df["nonpos_income"], "PWGTP"].sum()
w_total = df["PWGTP"].sum()

nonpos_share = w_nonpos / w_total
nonpos_share

np.float64(0.28739170890047466)

In [55]:
# since ~30% of the pop has non-positive income (kids, and some adults), we keep a non-pos incomme category.
# Fixed bins for positive income
fixed_bins = [0, 20000, 50000, 100000, 200000, np.inf]
fixed_labels = ["0-19k", "20-49k", "50-99k", "100-199k", "200k+"]

income_fixed = pd.cut(
    df["PINCP"],
    bins=fixed_bins,
    labels=fixed_labels,
    right=False
)

# Add explicit category for <=0
df["income_tier_fixed"] = income_fixed.astype("object")
df.loc[df["nonpos_income"], "income_tier_fixed"] = "<=0"
df["income_tier_fixed"] = df["income_tier_fixed"].astype("category")

df["income_tier_fixed"].value_counts(dropna=False)

income_tier_fixed
<=0         4160467
20-49k      3815560
0-19k       3623266
50-99k      2779274
100-199k    1128262
200k+        405564
Name: count, dtype: int64

In [57]:
# % check on weighted distribution of income tiers
income_fixed_dist = df.groupby("income_tier_fixed")["PWGTP"].sum()
income_fixed_dist = income_fixed_dist / income_fixed_dist.sum()
(income_fixed_dist * 100).round(2)

income_tier_fixed
0-19k       21.25
100-199k     6.76
20-49k      23.88
200k+        2.28
50-99k      17.09
<=0         28.74
Name: PWGTP, dtype: float64

In [58]:
# weighted percentile bins for positive income only + ,=0 category
def weighted_quantiles(x, w, qs):
    """
    x: 1D numpy array
    w: 1D numpy array (same length)
    qs: list/array of quantiles in [0,1]
    returns: quantile values at qs
    """
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    order = np.argsort(x)
    x_sorted = x[order]
    w_sorted = w[order]

    cw = np.cumsum(w_sorted)
    cw = cw / cw[-1]  # normalize to [0,1]

    return np.interp(qs, cw, x_sorted)

# Use only positive incomes for percentile computation
mask_pos = (df["PINCP"].notna()) & (df["PINCP"] > 0) & (df["PWGTP"].notna()) & (df["PWGTP"] > 0)

x = df.loc[mask_pos, "PINCP"].to_numpy()
w = df.loc[mask_pos, "PWGTP"].to_numpy()

# Quintiles (20/40/60/80)
q_vals = weighted_quantiles(x, w, qs=[0.2, 0.4, 0.6, 0.8])
q_vals

array([12300., 27000., 45000., 75000.])

In [59]:
pct_bins = [0, q_vals[0], q_vals[1], q_vals[2], q_vals[3], np.inf]
pct_labels = ["P0-20", "P20-40", "P40-60", "P60-80", "P80-100"]

income_pct = pd.cut(
    df["PINCP"],
    bins=pct_bins,
    labels=pct_labels,
    right=False
)

df["income_tier_pct"] = income_pct.astype("object")
df.loc[df["nonpos_income"], "income_tier_pct"] = "<=0"
df["income_tier_pct"] = df["income_tier_pct"].astype("category")

# Weighted distribution
income_pct_dist = df.groupby("income_tier_pct")["PWGTP"].sum()
income_pct_dist = income_pct_dist / income_pct_dist.sum()
(income_pct_dist * 100).round(2)

income_tier_pct
<=0        28.74
P0-20      14.23
P20-40     14.16
P40-60     13.87
P60-80     14.21
P80-100    14.79
Name: PWGTP, dtype: float64

In [60]:
# adult only distribution
(weighted_distribution(
    df[df["actor_class"] == "Adult"],
    "income_tier_fixed"
) * 100).round(2)

income_tier_fixed
0-19k       25.82
100-199k     8.68
20-49k      30.63
200k+        2.93
50-99k      21.96
<=0          9.98
Name: PWGTP, dtype: float64

In [62]:
# income tier features should be NA for minors, since they don't have personal income.
df["income_tier_fixed"] = np.where(
    df["actor_class"] == "Adult",
    df["income_tier_fixed"],
    pd.NA
)

df["income_tier_pct"] = np.where(
    df["actor_class"] == "Adult",
    df["income_tier_pct"],
    pd.NA
)

In [63]:
# adult only distribution
(weighted_distribution(
    df[df["actor_class"] == "Adult"],
    "income_tier_fixed"
) * 100).round(2)

income_tier_fixed
0-19k       25.82
100-199k     8.68
20-49k      30.63
200k+        2.93
50-99k      21.96
<=0          9.98
Name: PWGTP, dtype: float64

In [64]:
# Check minors
df.loc[df["actor_class"] == "Minor", "income_tier_fixed"].isna().all()

np.True_

### Income Tier Definition (Adult-Only)

Income tiers (`income_tier_fixed` and `income_tier_pct`) are defined only for Adults (AGEP ≥ 18).  
Minors are assigned NA because personal income (PINCP) reflects individual economic agency, which is not conceptually applicable to dependent children.

This avoids conflating non-earners (e.g., minors) with low-income adults and preserves structural interpretability for segmentation and modeling.

In [65]:
# inspecting basic race/ethnicity codes --> HISP>1 is hispanic; HISP==1 not hispanic.
df["RAC1P"].value_counts().head()
df["HISP"].value_counts().head()

HISP
1     13527233
2      1423908
3       214705
24      136965
4        99596
Name: count, dtype: int64

In [66]:
# creating race_eth label. Outputs: Hispanic, White_NH, Black_NH, Asian_NH, Other_NH

conditions = [
    df["HISP"] > 1,                                  # Hispanic (any race)
    (df["HISP"] == 1) & (df["RAC1P"] == 1),          # White NH
    (df["HISP"] == 1) & (df["RAC1P"] == 2),          # Black NH
    (df["HISP"] == 1) & (df["RAC1P"] == 6),          # Asian NH
]

choices = [
    "Hispanic",
    "White_NH",
    "Black_NH",
    "Asian_NH"
]

df["race_eth"] = np.select(
    conditions,
    choices,
    default="Other_NH"
)

df["race_eth"] = df["race_eth"].astype("category")

In [67]:
# weighted race_eth distribution check
race_distribution = (
    df.groupby("race_eth")["PWGTP"]
      .sum()
)

race_distribution = race_distribution / race_distribution.sum()
(race_distribution * 100).round(2)

race_eth
Asian_NH     5.75
Black_NH    12.02
Hispanic    18.99
Other_NH     5.06
White_NH    58.18
Name: PWGTP, dtype: float64

In [72]:
# creating marital status label
mar_array = np.select(
    [
        df["MAR"] == 1,
        df["MAR"].isin([2, 3, 4]),
        df["MAR"] == 5
    ],
    [
        "Married",
        "Previously_Married",
        "Never_Married"
    ],
    default=None
)

df["mar_tier"] = pd.Series(mar_array, index=df.index).astype("category")

In [73]:
(df.groupby("mar_tier")["PWGTP"].sum() /
 df["PWGTP"].sum() * 100).round(2)

mar_tier
Married               39.18
Never_Married         46.22
Previously_Married    14.60
Name: PWGTP, dtype: float64

In [74]:
# recoding commute mode based on JWTRNS variable
df["JWTRNS"].value_counts().sort_index().head(20)

JWTRNS
1.0     5674109
2.0      103571
3.0       85871
4.0       27507
5.0        6316
6.0        3018
7.0       12803
8.0        9792
9.0       32934
10.0     197603
11.0    1001746
12.0      75489
Name: count, dtype: int64

In [76]:
jw = df["JWTRNS"].astype("Int64")  # nullable int

# start as missing (not applicable)
df["commute_tier"] = pd.Series(pd.NA, index=df.index, dtype="string")

# map known modes
df.loc[jw == 11, "commute_tier"] = "Work_From_Home"
df.loc[jw.isin([1,2,3,4,5,6,7,8,9]), "commute_tier"] = "Car"
df.loc[jw == 10, "commute_tier"] = "Public_Transit"
df.loc[jw == 12, "commute_tier"] = "Walk"

# anything else that is non-missing becomes "Other"
df.loc[jw.notna() & df["commute_tier"].isna(), "commute_tier"] = "Other"

# compress to category
df["commute_tier"] = df["commute_tier"].astype("category")

In [77]:
df["commute_tier"].value_counts(dropna=False)

commute_tier
<NA>              8681634
Car               5955921
Work_From_Home    1001746
Public_Transit     197603
Walk                75489
Name: count, dtype: int64

In [78]:
# weighted commute distribution
(df.groupby("commute_tier")["PWGTP"].sum() / df["PWGTP"].sum() * 100).round(2)

commute_tier
Car               39.24
Public_Transit     1.14
Walk               0.53
Work_From_Home     6.47
Name: PWGTP, dtype: float64

In [79]:
# workers only commute distribution
(
    df[df["emp_tier"] == "Employed"]
      .groupby("commute_tier")["PWGTP"]
      .sum()
    /
    df[df["emp_tier"] == "Employed"]["PWGTP"].sum()
    * 100
).round(2)

commute_tier
Car               81.05
Public_Transit     2.35
Walk               1.09
Work_From_Home    13.36
Name: PWGTP, dtype: float64

### Commute Mode Harmonization (JWTRNS)

The ACS variable `JWTRNS` (Journey to Work – Transportation Mode) is recoded into a simplified `commute_tier` variable:

- Car  
- Public_Transit  
- Walk  
- Work_From_Home  
- Other  

This variable serves as a lifestyle and urbanicity proxy and is behaviorally relevant for media exposure, daily routines, and segmentation modeling.

Commute mode is defined for the full population; individuals for whom commuting is not applicable (e.g., non-workers, minors) retain NA.

In [81]:
# drop intermediate columns to save memory
struct_cols = [
    "SERIALNO",
    "SPORDER",
    "PWGTP",
    "actor_class",
    "age_bin",
    "sex_label",
    "race_eth",
    "edu_tier",
    "emp_tier",
    "income_tier_fixed",
    "income_tier_pct",
    "mar_tier",
    "commute_tier"
]

df_struct = df[struct_cols].copy()
df_struct.shape

(15912393, 13)

In [82]:
#normalize columns name (lowercase and snake_case)
import re

def to_snake(name):
    name = name.lower().strip()
    name = re.sub(r"[^\w]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name

df_struct.columns = [to_snake(c) for c in df_struct.columns]

df_struct.columns

Index(['serialno', 'sporder', 'pwgtp', 'actor_class', 'age_bin', 'sex_label',
       'race_eth', 'edu_tier', 'emp_tier', 'income_tier_fixed',
       'income_tier_pct', 'mar_tier', 'commute_tier'],
      dtype='str')

In [83]:
# save structural features dataframe
STRUCT_PATH = os.path.join(BASE_PATH, "mk_structural_person_v1_1.parquet")

df_struct.to_parquet(STRUCT_PATH, index=False)

STRUCT_PATH, df_struct.shape

('/Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/mk_structural_person_v1_1.parquet',
 (15912393, 13))

In [84]:
# Final memory cleanup
del df
import gc
gc.collect()

1671